<img src="https://images.seeklogo.com/logo-png/28/1/u-cayetano-heredia-logo-png_seeklogo-289967.png"
     alt="LOGO UPCH" width="250" align="right">

<br>

<h1><font color="#7F000E" size="5">UNIVERSIDAD PERUANA CAYETANO HEREDIA</font></h1>

<h1><font color="#7F000E" size="6">SEÑALES BIOMÉDICAS</font></h1>

<h1><font color="#7F000E" size="4">LABORATORIO 10 — SOLUCIONARIO: resultados explicados y respuestas</font></h1>

<br>

<div style="text-align:right">
  <font color="#7F000E" size="3">Ing. Alexander Valdez Portocarrero</font><br>
  <font color="#7F000E" size="3">Curso: Introducción a las Señales Biomédicas</font><br>
  <font color="#7F000E" size="3">Registro: OpenSignals BITalino, canal A1 (04/09/2026; 1000 Hz; 10 bits)</font><br>
</div>

## Cómo leer este solucionario

Este cuaderno conserva la estructura original del laboratorio y adapta el análisis al **único registro EMG** proporcionado. Después de cada resultado importante encontrarás:

| Marca | Qué contiene |
|---|---|
| 📘 **Explicación** | Qué hizo la celda anterior y cómo interpretar su salida |
| ✅ **Respuesta** | Respuesta a las preguntas de verificación y ejercicios |
| 🔎 **Verificación adicional** | Cálculos extra para sustentar la interpretación |

**Importante sobre los datos:** se recibió un único registro OpenSignals de 44.1 s (04/09/2026, 15:20:15.812; Fs = 1000 Hz; canal A1). Los dos TXT adjuntos eran copias del mismo registro. No existe una medición BASELINE independiente. Para ejecutar los apartados que necesitan una referencia de menor actividad, el cuaderno extrae automáticamente un segmento continuo de 2 s con menor RMS del mismo archivo. Las etiquetas de actividad/menor actividad son estimaciones algorítmicas, no anotaciones clínicas.

Los resultados numéricos se generan al ejecutar las celdas con el TXT original; no se conservan cifras del solucionario anterior cuando correspondían a otros archivos.

## Pregunta guía del laboratorio

> **¿Cómo convierte un dispositivo la actividad eléctrica de un músculo en números que puede entender?**

Una prótesis mioeléctrica, un exoesqueleto de rehabilitación o un sistema que detecta fatiga en un trabajador **no "miran" la señal cruda**: la resumen en un conjunto pequeño de números llamados **características** (*features*). Elegir y calcular bien esas características es, muchas veces, más importante que el clasificador que viene después.



> **Adaptación de datos:** este solucionario utiliza un solo registro EMG de 44.1 s. No se recibió un archivo BASELINE separado. La referencia de menor actividad se selecciona dentro de la misma grabación, por lo que las etiquetas y comparaciones son exploratorias.

### Objetivos de aprendizaje
Al terminar este laboratorio podrás:

1. Explicar el **origen fisiológico** de la sEMG y sus rangos típicos de amplitud y frecuencia.
2. **Convertir, filtrar y evaluar la calidad** de una señal real adquirida con BITalino.
3. **Segmentar** la señal en ventanas y **etiquetar** automáticamente reposo y contracción.
4. Calcular e interpretar características **temporales, estadísticas y espectrales**.
5. Construir una **matriz de características** y entrenar un clasificador **sin fuga de datos**.
6. Entender cómo una **CNN** aprende características a partir del **espectrograma (STFT)** y compararla con el enfoque clásico.

### Agenda (120 min)

| Bloque | Tiempo | Tipo |
|---|---|---|
| 0. Introducción y configuración | 5 min | Motivación |
| 1. Fundamentos fisiológicos | 15 min | Teoría + simulación |
| 2. Instrumentación y adquisición | 10 min | Técnica + práctica |
| 3. Preprocesamiento | 15 min | Práctica |
| 4. Segmentación y etiquetado | 10 min | Técnica + práctica |
| 5. Extracción de características clásicas | 25 min | Teoría + práctica |
| 6. Del vector de características al clasificador | 15 min | Práctica |
| 7. Deep learning: espectrograma + CNN | 15 min | Teoría + demo |
| 8. Ejercicios, cierre y referencias | 10 min | Síntesis |

**Datos del laboratorio:** dos registros del antebrazo/brazo tomados con BITalino:
- `BASELINEBRAZO.txt` — músculo en **reposo** (referencia de ruido).
- `PRUEBABRAZO1.txt` — **contracciones repetidas** separadas por reposo.

Si no tienes los archivos, el cuaderno genera automáticamente una **señal simulada** equivalente para que puedas seguir todo el flujo.

## 0. Configuración del entorno

Todas las bibliotecas usadas vienen preinstaladas en Google Colab (`numpy`, `scipy`, `pandas`, `matplotlib`, `scikit-learn`, `torch`). No es necesario `opensignalsreader`: escribiremos nuestro propio lector para **entender** el formato del archivo.

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import signal
from scipy.stats import kurtosis, skew

warnings.filterwarnings("ignore", category=RuntimeWarning)
np.random.seed(42)

plt.rcParams.update({
    "figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
})
COLOR_REPOSO, COLOR_CONTRAC, COLOR_UPCH = "#4C72B0", "#C44E52", "#7F000E"

try:
    from IPython.display import display
except ImportError:
    display = print

print("Entorno listo.")

📘 **Explicación.** Esta celda solo prepara el entorno: importa las bibliotecas, fija una semilla aleatoria (`np.random.seed(42)`) para que la simulación salga igual cada vez y define los colores de las gráficas. El mensaje *"Entorno listo."* confirma que todo se cargó sin errores.

---
# 1. Fundamentos fisiológicos (15 min)

### 1.1 La unidad motora
El músculo no se contrae "en bloque". Se organiza en **unidades motoras (UM)**: una **motoneurona** de la médula espinal y **todas las fibras musculares que inerva**. Cuando la motoneurona dispara, todas sus fibras se despolarizan casi a la vez y producen un **potencial de acción de unidad motora (MUAP)**: una forma de onda bifásica o trifásica de pocos milisegundos.

El sistema nervioso regula la fuerza con dos mecanismos:

| Mecanismo | Qué ocurre | Efecto en la sEMG |
|---|---|---|
| **Reclutamiento** | Se activan más unidades motoras (primero las pequeñas — *principio de Henneman*) | Aumenta la amplitud |
| **Codificación por frecuencia** | Cada UM dispara más rápido (≈ 8 a 30 disparos/s) | Aumenta amplitud y densidad de la señal |

### 1.2 ¿Qué registra un electrodo de superficie?
Un electrodo sobre la piel no ve una UM aislada: ve la **suma espacial y temporal** de los MUAPs de muchas unidades, filtrada por grasa, piel y la propia distancia. El resultado es un **patrón de interferencia** que parece ruido aleatorio, pero cuya **amplitud** y **contenido espectral** codifican información sobre la activación muscular.

> 💡 **Idea clave:** la sEMG es como escuchar un estadio lleno desde afuera. No entiendes a ninguna persona en particular, pero sí sabes si hay pocos o muchos gritando (amplitud) y si las voces son agudas o graves (frecuencia).

### 1.3 Rangos típicos de la sEMG

| Propiedad | Valor típico |
|---|---|
| Amplitud | 0 – 10 mV pico a pico (en reposo: microvoltios) |
| Banda útil | ≈ 10 / 20 – 450 / 500 Hz |
| Energía dominante | ≈ 50 – 150 Hz |
| Distribución de amplitudes | Aproximadamente gaussiana de media cero en contracción sostenida |

Factores que la modifican: grosor del tejido adiposo, posición y distancia entre electrodos, temperatura, **fatiga** (desplaza el espectro a frecuencias más bajas), impedancia piel-electrodo y artefactos.

Veamos cómo surge el patrón de interferencia con una **simulación** simple.

In [ ]:
# Simulación didáctica: suma de trenes de MUAPs
FS_SIM = 2000
t_sim = np.arange(0, 1.0, 1/FS_SIM)

def muap(duracion_ms=10, fs=FS_SIM):
    # Forma bifásica tipo "derivada de gaussiana" (modelo simplificado de MUAP)
    tt = np.arange(-duracion_ms/2, duracion_ms/2, 1000/fs) / 1000
    s = duracion_ms / 1000 / 6
    return -tt / s * np.exp(-tt**2 / (2*s**2))

def tren_um(fr_hz, amplitud, dur_ms, fs=FS_SIM, n=len(t_sim)):
    # Una unidad motora disparando a ~fr_hz con variabilidad del 20 % entre disparos
    x = np.zeros(n); k = np.random.uniform(0, 1/fr_hz)
    while k < n/fs:
        x[int(k*fs)] = 1
        k += np.random.normal(1/fr_hz, 0.2/fr_hz)
    return amplitud * np.convolve(x, muap(dur_ms, fs), mode="same")

fig, ax = plt.subplots(3, 1, figsize=(13, 6.5), sharex=True)
for a, n_um in zip(ax, [1, 5, 60]):
    total = sum(tren_um(np.random.uniform(8, 25), np.random.uniform(0.2, 1.0),
                        np.random.uniform(6, 14)) for _ in range(n_um))
    a.plot(t_sim, total, lw=0.7, color=COLOR_UPCH)
    a.set_title(f"{n_um} unidad(es) motora(s) activa(s)", loc="left", fontsize=10)
    a.set_ylabel("u.a.")
ax[-1].set_xlabel("Tiempo (s)")
plt.suptitle("De MUAPs individuales al patrón de interferencia de la sEMG", y=1.0)
plt.tight_layout(); plt.show()

📘 **Explicación.** La simulación construye la sEMG "desde abajo":
1. Se crea la forma de **un** MUAP (una onda que sube y baja en pocos milisegundos).
2. Cada unidad motora "dispara" ese MUAP entre 8 y 25 veces por segundo, con algo de irregularidad.
3. Se **suman** los trenes de 1, 5 y 60 unidades motoras.

En la gráfica de arriba (1 UM) se ven picos separados y regulares, uno por cada disparo. En la del medio (5 UM) los picos empiezan a mezclarse. En la de abajo (60 UM) ya no se distingue ningún disparo: la señal parece ruido. Además, el eje vertical crece de una gráfica a otra: **más unidades activas producen más amplitud**.

> ❓ **Preguntas de verificación**
> 1. Con 1 unidad motora, ¿puedes distinguir cada disparo? ¿Y con 60?
> 2. ¿Qué mecanismo fisiológico representa pasar de 5 a 60 unidades?
> 3. Si la señal con 60 UM "parece ruido", ¿por qué sigue siendo útil? ¿Qué cambia entre las tres gráficas que sí podríamos medir?

✅ **Respuestas**

1. **Con 1 UM, sí:** cada disparo aparece como un pico aislado, separado del siguiente. **Con 60 UM, no:** los MUAPs se superponen en el tiempo y forman un patrón continuo (patrón de interferencia) donde ningún disparo individual es reconocible.
2. Representa el **reclutamiento**: el sistema nervioso activa más unidades motoras para producir más fuerza.
3. Aunque no podamos ver cada disparo, la señal sigue llevando información en **propiedades globales** que sí se pueden medir. La principal es la **amplitud**: en la figura, el rango del eje vertical crece al pasar de 1 a 5 y a 60 UM, y eso lo captura una medida como el RMS. También cambia la **densidad** de la señal (cuántas veces sube y baja por segundo), que capturan medidas como los cruces por cero. Esas propiedades son justamente las **características** que calcularemos en el Bloque 5.

---
# 2. Instrumentación y adquisición (10 min)

### 2.1 Cadena de adquisición del BITalino
```
Músculo → electrodos Ag/AgCl → amplificador diferencial (G = 1009) → filtro analógico → ADC 10 bits → Bluetooth → OpenSignals (.txt)
```

- **Configuración bipolar / diferencial:** dos electrodos sobre el vientre muscular y uno de referencia en zona eléctricamente neutra (hueso). El amplificador **resta** ambos electrodos: lo que es común a los dos (p. ej., interferencia de la red eléctrica) se cancela (*rechazo en modo común*, CMRR).
- **Colocación:** recomendaciones **SENIAM** — electrodos alineados con las fibras, entre el punto motor y el tendón, con ~20 mm entre centros; piel limpia para reducir la impedancia.

### 2.2 Muestreo y cuantización
- **Frecuencia de muestreo:** el archivo indica **Fs = 1000 Hz** → frecuencia de Nyquist = 500 Hz. Cubre la banda útil de la sEMG.
- **Resolución:** 10 bits → el ADC entrega enteros de **0 a 1023**. El valor ~512 corresponde a 0 V.

### 2.3 Conversión de unidades (hoja técnica del sensor EMG BITalino)

$$
\text{EMG}_{mV} = \frac{\left(\dfrac{ADC}{2^{n}} - \dfrac{1}{2}\right)\cdot V_{CC}}{G_{EMG}} \times 1000,
\qquad V_{CC}=3.3\ \text{V},\ G_{EMG}=1009,\ n = 10
$$

Esto da un rango de medición de aproximadamente **±1.64 mV**. Cualquier actividad mayor **satura** el ADC (se queda pegada en 0 o en 1023).

### 2.4 Fuentes de ruido y artefactos

| Fuente | Característica | Cómo se trata |
|---|---|---|
| Red eléctrica | 60 Hz en Perú (y armónicos) | Diferencial + filtro *notch* |
| Movimiento de electrodos/cables | < 20 Hz, gran amplitud | Pasa-altos ≈ 20 Hz, buena fijación |
| ECG (cerca del tórax) | Picos periódicos ~1 Hz | Pasa-altos, ubicación |
| Ruido térmico/electrónico | Banda ancha, bajo nivel | Se estima con un registro en menor actividad |
| **Saturación (clipping)** | Señal "aplanada" en el máximo/mínimo del ADC | Prevención (ganancia) y detección |

> 💡 **Idea clave:** ningún filtro digital recupera información que se perdió por saturación. La calidad empieza en la adquisición.

### 2.5 Carga de los archivos

El formato de texto de OpenSignals tiene tres partes:
1. Línea `# OpenSignals Text File Format`.
2. Línea `# {...}` con un **JSON** de metadatos: frecuencia de muestreo, resolución, sensores, nombres de columnas.
3. `# EndOfHeader` y luego los datos en columnas separadas por tabulaciones (`nSeq`, entradas digitales, `A1`...).

Escribimos un lector que **lee los metadatos del propio archivo** en lugar de suponerlos.

**En Colab:** ejecuta la celda siguiente; si los archivos no están en la carpeta, se abrirá un diálogo para subirlos.

In [ ]:
# Un único registro EMG: no se cuenta con un archivo BASELINE independiente.
RUTA_REGISTRO = "opensignals_0C4314247FC7_2026-09-04_15-20-06 (1)(1).txt"

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB and not os.path.exists(RUTA_REGISTRO):
    from google.colab import files
    print("Sube el archivo TXT del único registro EMG.")
    archivos = files.upload()
    if archivos:
        RUTA_REGISTRO = next(iter(archivos))

assert os.path.exists(RUTA_REGISTRO), (
    "No se encontró el TXT. Súbelo a Colab o colócalo junto al notebook."
)
USAR_SIMULADA = False
print("Usando señal REAL (BITalino) — registro único")


📘 **Explicación.** El cuaderno carga un único registro real de BITalino. No se generan datos simulados ni se espera un segundo archivo.

In [ ]:
def leer_opensignals(ruta, etiqueta_canal="A1"):
    # Lee los valores ADC y los metadatos del canal indicado.
    with open(ruta, "r", encoding="utf-8") as f:
        lineas = f.readlines()
    meta = list(json.loads(lineas[1].lstrip("# ").strip()).values())[0]
    datos = np.loadtxt(ruta, comments="#")
    idx_col = meta["column"].index(etiqueta_canal)
    meta["bits"] = meta["resolution"][idx_col]
    return datos[:, idx_col].astype(float), meta

adc_prueba, meta_prueba = leer_opensignals(RUTA_REGISTRO)
FS = int(meta_prueba["sampling rate"])
N_BITS = int(meta_prueba["bits"])

# Referencia intra-registro: segmento continuo de 2 s con menor RMS.
# NO es un baseline independiente ni una medición de reposo controlada.
VCC, G_EMG = 3.3, 1009
def adc_a_mV(adc, n_bits=N_BITS, vcc=VCC, ganancia=G_EMG):
    return ((adc / 2**n_bits) - 0.5) * vcc / ganancia * 1000

emg_prueba_mV = adc_a_mV(adc_prueba)
dur_ref = min(2.0, max(0.5, len(adc_prueba)/FS/10))
n_ref = int(dur_ref * FS)
rms_ref = np.sqrt(np.convolve(emg_prueba_mV**2, np.ones(n_ref)/n_ref, mode="valid"))
i_ref = int(np.argmin(rms_ref))
adc_base = adc_prueba[i_ref:i_ref+n_ref].copy()
emg_base_mV = emg_prueba_mV[i_ref:i_ref+n_ref].copy()
meta_base = dict(meta_prueba)
t_base = np.arange(len(emg_base_mV)) / FS
t_prueba = np.arange(len(emg_prueba_mV)) / FS

print(f"Registro: {meta_prueba.get('date')} {meta_prueba.get('time')}")
print(f"Sensor: {meta_prueba.get('sensor')} | Fs = {FS} Hz | resolución = {N_BITS} bits")
print(f"Duración del registro: {len(adc_prueba)/FS:.2f} s")
print(f"Referencia intra-registro: {i_ref/FS:.2f}–{(i_ref+n_ref)/FS:.2f} s ({dur_ref:.1f} s)")
print("Aviso: la referencia se extrajo del mismo registro; no equivale a un BASELINE independiente.")


In [ ]:
N_BITS

📘 **Explicación.** El lector obtuvo los metadatos directamente del único archivo OpenSignals:

- Sensor **EMGBITREV**, canal **A1**, **Fs = 1000 Hz**, resolución de **10 bits** (0–1023).
- Registro del **04/09/2026**, hora **15:20:15.812**, con **44 100 muestras** (44.1 s).
- El ADC recorre desde **0 hasta 1023**, por lo que hay muestras en los límites del conversor.
- Para conservar los cálculos que requieren una referencia de reposo, se seleccionó automáticamente un segmento continuo de 2 s con menor RMS dentro del mismo registro. Este segmento se denomina **referencia intra-registro** y no debe interpretarse como un baseline independiente.


In [ ]:
VCC, G_EMG = 3.3, 1009

def adc_a_mV(adc, n_bits=N_BITS, vcc=VCC, ganancia=G_EMG):
    return ((adc / 2**n_bits) - 0.5) * vcc / ganancia * 1000

# Conversión del registro completo y del segmento de referencia intra-registro
emg_base_mV = adc_a_mV(adc_base)
emg_prueba_mV = adc_a_mV(adc_prueba)
t_base = np.arange(len(emg_base_mV)) / FS
t_prueba = np.arange(len(emg_prueba_mV)) / FS

rango_max = adc_a_mV(2**N_BITS - 1)
print(f"Rango medible del sensor: ±{rango_max:.3f} mV")
print(f"Resolución (1 cuenta ADC): {adc_a_mV(513) - adc_a_mV(512):.4f} mV = "
      f"{(adc_a_mV(513) - adc_a_mV(512))*1000:.2f} µV")

fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=False, sharey=True)
ax[0].plot(t_base, emg_base_mV, lw=0.5, color=COLOR_REPOSO)
ax[0].set_title("REFERENCIA INTRA-REGISTRO — menor actividad (2 s)", loc="left")
ax[1].plot(t_prueba, emg_prueba_mV, lw=0.5, color=COLOR_CONTRAC)
ax[1].set_title("REGISTRO COMPLETO — canal A1", loc="left")
for a in ax:
    a.axhline(rango_max, ls="--", color="k", lw=0.8); a.axhline(-rango_max, ls="--", color="k", lw=0.8)
    a.set_ylabel("EMG (mV)")
ax[1].set_xlabel("Tiempo (s)")
plt.tight_layout(); plt.show()


📘 **Explicación.** La conversión usa la ecuación del sensor BITalino. El rango aproximado es ±1.63 mV y la resolución es cercana a 3.2 µV por cuenta ADC. La primera gráfica corresponde a un segmento de menor actividad extraído del propio registro; la segunda muestra los 44.1 s completos. No son dos adquisiciones independientes.

### 2.6 Control de calidad: ¿hay saturación?
Una muestra está saturada si su valor ADC es exactamente 0 o $2^n-1$. Es importante **cuantificarlo** antes de calcular características, porque recorta los picos y distorsiona sobre todo las medidas sensibles a los extremos (curtosis, varianza).

In [ ]:
sat_prueba = (adc_prueba <= 0) | (adc_prueba >= 2**N_BITS - 1)
sat_base   = (adc_base   <= 0) | (adc_base   >= 2**N_BITS - 1)
print(f"Saturación BASELINE: {sat_base.sum()} muestras ({100*sat_base.mean():.3f} %)")
print(f"Saturación PRUEBA  : {sat_prueba.sum()} muestras ({100*sat_prueba.mean():.3f} %)")

if sat_prueba.any():
    i0 = np.flatnonzero(sat_prueba)[0]
    sl = slice(max(0, i0 - 150), i0 + 150)
    plt.figure(figsize=(12, 3))
    plt.plot(t_prueba[sl], emg_prueba_mV[sl], lw=0.9, color=COLOR_CONTRAC)
    plt.plot(t_prueba[sl][sat_prueba[sl]], emg_prueba_mV[sl][sat_prueba[sl]], "ko", ms=3, label="saturada")
    plt.title("Zoom a un evento de saturación"); plt.xlabel("Tiempo (s)"); plt.ylabel("mV"); plt.legend()
    plt.show()

📘 **Explicación.** La saturación estricta se identifica cuando el ADC alcanza 0 o 1023. También se revisa 1022 como posible techo práctico. Cualquier saturación recorta la señal y afecta especialmente las características de amplitud y forma. La referencia intra-registro se extrae del mismo archivo y no permite estimar por separado el ruido instrumental.

🔎 **Verificación adicional — ¿dónde se "pega" realmente el ADC?**

La celda siguiente cuenta cuántas veces aparece cada valor extremo del ADC en la PRUEBA.

In [ ]:

# Verificación adicional: ¿en qué valores se "pega" realmente el ADC?
valores, cuentas = np.unique(adc_prueba, return_counts=True)
print("5 valores ADC más bajos :", [(int(v), int(c)) for v, c in zip(valores[:5], cuentas[:5])])
print("5 valores ADC más altos :", [(int(v), int(c)) for v, c in zip(valores[-5:], cuentas[-5:])])

# Criterio corregido: también se cuenta el valor 1022 (2^n - 2), donde se observa la meseta superior
sat_corr = (adc_prueba <= 0) | (adc_prueba >= 2**N_BITS - 2)
n_win_250 = int(0.25 * FS)
ini_250 = np.arange(0, len(adc_prueba) - n_win_250 + 1, n_win_250 // 2)
print(f"\nCriterio original (0 o 1023)       : {sat_prueba.sum()} muestras ({100*sat_prueba.mean():.3f} %)")
print(f"Criterio corregido (0, 1022 o 1023): {sat_corr.sum()} muestras ({100*sat_corr.mean():.3f} %)")
print(f"Ventanas de 250 ms con saturación (criterio corregido): "
      f"{sum(sat_corr[i:i+n_win_250].any() for i in ini_250)}")


📘 **Resultado de la verificación.** El conteo de valores extremos se calcula directamente sobre el único registro. Observa si se acumulan muestras en 0, 1022 o 1023. Ese comportamiento sirve para identificar posible saturación, pero no debe confundirse con una medición de baseline.

> ❓ **Preguntas de verificación**
> 1. ¿Qué porcentaje de muestras está saturado?
> 2. ¿Qué cambiarías en la adquisición para evitarlo?
> 3. ¿Por qué un registro de menor actividad dentro de la misma grabación no equivale a un baseline independiente?
>
> ✅ **Respuestas**
>
> 1. El porcentaje se imprime en la celda anterior. Se debe reportar tanto el criterio estricto (0 o 1023) como el criterio ampliado si se observa acumulación en 1022.
> 2. Reducir la ganancia o utilizar un sistema con mayor rango de entrada, revisar la colocación de electrodos y controlar la intensidad de la contracción. La saturación no puede corregirse con un filtro posterior.
> 3. Porque se obtuvo durante la misma sesión y con la misma colocación/condiciones de adquisición. No constituye una medición separada de reposo ni permite aislar de forma rigurosa el ruido instrumental.

---
# 3. Preprocesamiento (15 min)

### 3.1 ¿Qué queremos eliminar?
1. **Componente continua (offset)** y derivas lentas → el pasa-altos lo resuelve.
2. **Artefactos de movimiento** (< 20 Hz).
3. **Ruido de alta frecuencia** por encima de la banda útil.
4. **Interferencia de 60 Hz** de la red eléctrica.

### 3.2 Diseño de filtros
- **Pasa-banda Butterworth 20–450 Hz, orden 4:** respuesta plana en la banda de paso. Se implementa en forma **SOS** (*second-order sections*) por estabilidad numérica.
- **Notch IIR en 60 Hz, Q = 30:** elimina una banda angosta (~2 Hz) alrededor de 60 Hz.
- **Filtrado de fase cero (`sosfiltfilt` / `filtfilt`):** filtra hacia adelante y hacia atrás, sin desfase. Válido para análisis **offline**; un sistema en tiempo real (prótesis) usaría filtros causales.

> ⚠️ Condición obligatoria: $0 < f_{baja} < f_{alta} < F_s/2$.

Antes de filtrar, miremos el **espectro** (densidad espectral de potencia, PSD) estimado con el **método de Welch**, que promedia periodogramas de segmentos solapados y da una estimación más estable que una sola FFT.

In [ ]:
def psd(x, fs=FS, nperseg=1024):
    return signal.welch(x - np.mean(x), fs=fs, nperseg=nperseg)

f_b, P_b = psd(emg_base_mV)
f_p, P_p = psd(emg_prueba_mV)

plt.figure(figsize=(12, 4))
plt.semilogy(f_b, P_b, label="referencia intra-registro (menor actividad)", color=COLOR_REPOSO)
plt.semilogy(f_p, P_p, label="PRUEBA (contracciones)", color=COLOR_CONTRAC)
for k in range(1, 9):
    plt.axvline(60*k, color="gray", ls=":", lw=0.8)
plt.axvspan(20, 450, color="green", alpha=0.06, label="banda 20–450 Hz")
plt.xlabel("Frecuencia (Hz)"); plt.ylabel("PSD (mV²/Hz)")
plt.title("Espectro de la señal cruda (líneas punteadas: 60 Hz y armónicos)")
plt.legend(); plt.show()



###**La PSD por el método de Welch (Power Spectral Density)**

La PSD por el método de Welch (Power Spectral Density) permite conocer cómo se distribuye la potencia de una señal entre sus diferentes frecuencias.

¿Cómo funciona?
1. Divide la señal en segmentos solapados.
2. Aplica una ventana a cada segmento para reducir la fuga espectral.
3. Calcula el periodograma de cada segmento mediante la FFT.
4. Promedia los periodogramas para obtener una estimación más estable de la PSD.
¿Para qué sirve en una señal sEMG?

Permite identificar las frecuencias donde se concentra la potencia de la actividad eléctrica muscular y calcular características como la frecuencia media y la frecuencia mediana, útiles para estudiar cambios en la señal, por ejemplo, durante la fatiga muscular.

En resumen: Welch transforma el análisis de una señal en el tiempo en una estimación de su distribución de potencia en frecuencia, reduciendo las fluctuaciones que tendría un único periodograma.

📘 **Explicación.** Esta gráfica es el **espectro** (PSD por Welch): indica cuánta potencia tiene la señal en cada frecuencia. El eje vertical es logarítmico; cada división es un factor de 10.

- **referencia intra-registro (azul):** potencia muy baja y con **picos angostos en 60 Hz y 120 Hz**, que es la interferencia de la red eléctrica y su segundo armónico. También aparece un pico cerca de 400 Hz, que no es múltiplo de 60 Hz; con estos datos no podemos determinar su origen.
- **PRUEBA (rojo):** está entre **~100 y ~10 000 veces por encima** del referencia intra-registro (de 2 a 4 divisiones del eje logarítmico), con la mayor diferencia en las frecuencias bajas. La potencia es máxima por debajo de ~100 Hz y cae de forma progresiva hacia las frecuencias altas. Es la forma típica del espectro de la sEMG.
- En la prueba **no se ve un pico claro de 60 Hz**: la actividad muscular es tan grande que lo "tapa".

In [ ]:
def filtrar_emg(x, fs=FS, f_baja=20, f_alta=450, orden=4, f_notch=60, q_notch=30):
    if not (0 < f_baja < f_alta < fs/2):
        raise ValueError(f"Se requiere 0 < f_baja < f_alta < Fs/2 (Fs = {fs} Hz).")
    sos = signal.butter(orden, [f_baja, f_alta], btype="bandpass", fs=fs, output="sos")
    y = signal.sosfiltfilt(sos, x - np.mean(x))
    if f_notch:
        b, a = signal.iirnotch(f_notch, q_notch, fs=fs)
        y = signal.filtfilt(b, a, y)
    return y

# Respuesta en frecuencia de los filtros
sos = signal.butter(4, [20, 450], btype="bandpass", fs=FS, output="sos")
w, h_bp = signal.sosfreqz(sos, worN=4096, fs=FS)
b_n, a_n = signal.iirnotch(60, 30, fs=FS)
_, h_n = signal.freqz(b_n, a_n, worN=4096, fs=FS)
plt.figure(figsize=(12, 3.2))
plt.plot(w, 20*np.log10(np.abs(h_bp*h_n) + 1e-12), color=COLOR_UPCH)
plt.ylim(-60, 5); plt.xlabel("Frecuencia (Hz)"); plt.ylabel("Ganancia (dB)")
plt.title("Respuesta en frecuencia: pasa-banda 20–450 Hz + notch 60 Hz"); plt.show()

emg_base_f   = filtrar_emg(emg_base_mV)
emg_prueba_f = filtrar_emg(emg_prueba_mV)

📘 **Explicación.** La gráfica muestra cómo trata el filtro a cada frecuencia (0 dB = la deja pasar intacta; valores negativos = la atenúa):

- Entre ~25 Hz y ~430 Hz la curva está en 0 dB: **la banda útil de la sEMG pasa sin cambios**.
- Por debajo de 20 Hz y por encima de 450 Hz la curva cae rápidamente: se eliminan las derivas lentas, los artefactos de movimiento y el ruido de alta frecuencia.
- En **60 Hz** aparece una muesca angosta: es el filtro *notch*, que elimina solo la interferencia de la red eléctrica sin afectar las frecuencias vecinas.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
f1, P1 = psd(emg_prueba_mV); f2, P2 = psd(emg_prueba_f)
ax[0].semilogy(f1, P1, label="cruda", alpha=0.7); ax[0].semilogy(f2, P2, label="filtrada")
ax[0].set_title("PSD — PRUEBA antes/después"); ax[0].set_xlabel("Hz"); ax[0].legend()

sl = (t_prueba > 5) & (t_prueba < 9)
ax[1].plot(t_prueba[sl], emg_prueba_mV[sl], lw=0.5, alpha=0.6, label="cruda")
ax[1].plot(t_prueba[sl], emg_prueba_f[sl], lw=0.5, label="filtrada")
ax[1].set_title("Dominio del tiempo (zoom 5–9 s)"); ax[1].set_xlabel("s"); ax[1].set_ylabel("mV"); ax[1].legend()
plt.tight_layout(); plt.show()

# Estimación simple de la relación señal-ruido usando el baseline como piso de ruido
rms = lambda x: np.sqrt(np.mean(x**2))
p95 = np.percentile(np.abs(emg_prueba_f), 95)
print(f"RMS del baseline filtrado (piso de ruido): {rms(emg_base_f)*1000:.2f} µV")
print(f"RMS global de la prueba filtrada        : {rms(emg_prueba_f)*1000:.2f} µV")
print(f"SNR aproximada = 20·log10(RMS_prueba / RMS_ruido) = "
      f"{20*np.log10(rms(emg_prueba_f)/rms(emg_base_f)):.1f} dB")

📘 **Explicación.** La comparación de PSD y de la señal en el tiempo permite observar el efecto del filtrado sobre este registro. El segmento de referencia intra-registro se extrajo de la misma grabación y sirve únicamente como aproximación de menor actividad. **No es correcto interpretarlo como ruido instrumental medido en un baseline independiente ni calcular una SNR experimental entre dos adquisiciones.**

🔎 **Verificación adicional — tamaño del pico de 60 Hz y ubicación de la energía de la contracción.**

Para responder con números las preguntas del Bloque 3, la celda siguiente calcula:
1. Cuántas veces es mayor la potencia en 60 Hz que la de sus frecuencias vecinas, antes y después de filtrar.
2. El espectro **solo de los tramos de contracción**, usando el mismo criterio de detección del Bloque 4.

In [ ]:

# Verificación adicional: tamaño del pico de 60 Hz y dónde está la energía de la actividad
def pico_relativo(x, f0=60):
    f, p = signal.welch(x - np.mean(x), fs=FS, nperseg=1024)
    vecinos = p[((f > f0 - 10) & (f < f0 - 3)) | ((f > f0 + 3) & (f < f0 + 10))]
    return p[np.argmin(np.abs(f - f0))] / np.median(vecinos)

for nombre, cruda, filtrada in [("referencia intra-registro", emg_base_mV, emg_base_f), ("PRUEBA", emg_prueba_mV, emg_prueba_f)]:
    print(f"{nombre:8s}: potencia en 60 Hz / potencia de frecuencias vecinas -> "
          f"cruda: {pico_relativo(cruda):6.1f} veces | filtrada: {pico_relativo(filtrada):5.2f} veces")

# Espectro solo de los tramos de actividad (usa la detección del bloque 4; se calcula aquí por adelantado)
env_tmp = np.sqrt(np.convolve(emg_prueba_f**2, np.ones(200)/200, mode="same"))
ref_tmp = env_tmp[env_tmp <= np.percentile(env_tmp, 20)]
activo_tmp = env_tmp > ref_tmp.mean() + 5 * ref_tmp.std()
f_c, P_c = signal.welch(emg_prueba_f[activo_tmp], fs=FS, nperseg=1024)
frac = lambda lo, hi: P_c[(f_c >= lo) & (f_c < hi)].sum() / P_c.sum()
acum = np.cumsum(P_c)
print(f"\nTramos de actividad: pico espectral en {f_c[np.argmax(P_c)]:.0f} Hz | "
      f"frecuencia mediana {f_c[np.searchsorted(acum, acum[-1]/2)]:.0f} Hz")
print(f"Fracción de potencia: 20–150 Hz = {100*frac(20,150):.1f} % | 150–450 Hz = {100*frac(150,451):.1f} % | "
      f"por debajo de 50 Hz = {100*frac(0,50):.1f} %")


> ❓ **Preguntas de verificación**
> 1. ¿Era visible el pico de 60 Hz en el espectro crudo? ¿Disminuyó tras el notch?
> 2. ¿En qué rango de frecuencias se concentra la energía de este registro?
> 3. ¿Por qué no se recomienda usar un pasa-bajos de 50 Hz para limpiar la sEMG?
>
> ✅ **Respuestas**
>
> 1. La presencia de un pico de 60 Hz y su atenuación deben comprobarse en las curvas PSD antes/después que genera el cuaderno. No se debe afirmar que desapareció si la gráfica no lo demuestra.
> 2. El rango de energía se obtiene de la PSD y de los cálculos de frecuencia mediana/media de las celdas. La sEMG suele contener información útil por encima de 50 Hz.
> 3. Porque un pasa-bajos de 50 Hz eliminaría una parte importante del contenido fisiológico de la sEMG, que suele extenderse aproximadamente hasta 450–500 Hz.

---
# 4. Segmentación y etiquetado (10 min)

### 4.1 Envolvente y detección de actividad (*onset*)
La **envolvente RMS móvil** resume la amplitud instantánea:

$$
\text{RMS}[n] = \sqrt{\frac{1}{W}\sum_{k=n-W/2}^{n+W/2} x[k]^2}
$$

Un método clásico de detección de *onset* compara la envolvente con un **umbral** derivado del reposo:

$$
\text{umbral} = \mu_{reposo} + k\,\sigma_{reposo}
$$

Probaremos primero tomando el reposo del **archivo referencia intra-registro**.

In [ ]:
def envolvente_rms(x, fs=FS, ventana_ms=200):
    w = max(1, int(fs * ventana_ms / 1000))
    return np.sqrt(np.convolve(x**2, np.ones(w)/w, mode="same"))

env_base   = envolvente_rms(emg_base_f)
env_prueba = envolvente_rms(emg_prueba_f)

K = 5
umbral_baseline = env_base.mean() + K * env_base.std()
print(f"Umbral con referencia intra-registro (2 s): {umbral_baseline*1000:.1f} µV")
print(f"Fracción del registro marcada como activa: {100*np.mean(env_prueba > umbral_baseline):.1f} %")


📘 **Explicación.** Se calculó la envolvente RMS de la referencia intra-registro de menor actividad y del registro completo. El umbral de media + 5 desviaciones estándar se usa como primera aproximación; al proceder de un segmento de solo 2 s de la misma grabación, debe interpretarse con cautela. La siguiente sección calcula además un umbral robusto usando el 20 % de menor envolvente del registro completo.

### 4.2 Una sorpresa (y una lección)
Con el umbral del referencia intra-registro, una gran parte de la prueba aparece "activa", incluso entre contracciones. Cuando eso ocurre suele deberse a que **el "reposo" dentro de la prueba no es igual al referencia intra-registro**: el músculo no se relaja del todo entre repeticiones (tono residual), los electrodos se movieron o la impedancia cambió entre registros.

> 💡 **Idea clave:** la referencia de reposo debe tomarse en **las mismas condiciones** que la señal a analizar.

Solución práctica: estimar el reposo **dentro del propio registro**, usando el 20 % de muestras con menor envolvente.

In [ ]:
ref = env_prueba[env_prueba <= np.percentile(env_prueba, 20)]
umbral = ref.mean() + K * ref.std()
print(f"Umbral con reposo intra-registro: {umbral*1000:.1f} µV")

def limpiar_mascara(activo, fs=FS, hueco_min_s=0.25, dur_min_s=0.30):
    # Une segmentos separados por huecos cortos y elimina activaciones demasiado breves
    d = np.diff(np.r_[0, activo.astype(int), 0])
    inicios, fines = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    segs = []
    for s, e in zip(inicios, fines):
        if segs and s - segs[-1][1] < hueco_min_s * fs:
            segs[-1] = (segs[-1][0], e)
        else:
            segs.append((s, e))
    segs = [(s, e) for s, e in segs if (e - s) >= dur_min_s * fs]
    mascara = np.zeros(len(activo), dtype=bool)
    for s, e in segs:
        mascara[s:e] = True
    return mascara, segs

mascara_activa, contracciones = limpiar_mascara(env_prueba > umbral)
dur = np.array([(e - s) / FS for s, e in contracciones])
print(f"Contracciones detectadas: {len(contracciones)} | duración media {dur.mean():.2f} ± {dur.std():.2f} s")

plt.figure(figsize=(14, 4))
plt.plot(t_prueba, emg_prueba_f, lw=0.4, color="gray", alpha=0.6, label="EMG filtrada")
plt.plot(t_prueba, env_prueba, lw=1.2, color=COLOR_UPCH, label="envolvente RMS (200 ms)")
plt.axhline(umbral, color="k", ls="--", lw=1, label="umbral")
for i, (s, e) in enumerate(contracciones):
    plt.axvspan(s/FS, e/FS, color=COLOR_CONTRAC, alpha=0.12)
    plt.text((s+e)/2/FS, env_prueba.max()*1.02, str(i+1), ha="center", fontsize=7)
plt.xlabel("Tiempo (s)"); plt.ylabel("mV"); plt.title("Contracciones detectadas automáticamente")
plt.legend(loc="lower right"); plt.show()

📘 **Explicación.** Para evitar depender exclusivamente del segmento de referencia de 2 s, se estima un umbral usando el 20 % de muestras con menor envolvente RMS del propio registro. La segmentación resultante es exploratoria: los segmentos detectados son candidatos a actividad muscular y pueden incluir errores por variaciones de esfuerzo, movimiento o ruido. El número y duración se obtienen de la salida de la celda anterior.

### 4.3 Ventaneo
Las características se calculan sobre **ventanas** cortas, porque la sEMG es **no estacionaria**: sus propiedades cambian en el tiempo.

| Parámetro | Valor | Compromiso |
|---|---|---|
| Longitud de ventana | 250 ms | Más corta → mejor resolución temporal, pero estimaciones más ruidosas. En control de prótesis se suele trabajar con ventanas de 100–300 ms por la restricción de latencia. |
| Solapamiento | 50 % | Más observaciones, pero **no independientes** entre sí. |

**Etiquetado de cada ventana:**
- `contraccion` si ≥ 80 % de sus muestras están dentro de una contracción detectada.
- `reposo` si ≤ 5 %.
- `transicion` en otro caso (se excluye del entrenamiento: su etiqueta es ambigua).

Además guardamos un **identificador de grupo** (`ciclo`): el reposo previo a la contracción *i* y la propia contracción *i* forman el ciclo *i*. Esto será clave en el Bloque 6 para evitar la **fuga de datos**.

In [ ]:
def crear_ventanas(x, fs=FS, ventana_ms=250, solapamiento=0.5):
    n = int(round(fs * ventana_ms / 1000))
    paso = max(1, int(round(n * (1 - solapamiento))))
    inicios = np.arange(0, len(x) - n + 1, paso)
    return np.stack([x[i:i+n] for i in inicios]), inicios

VENTANA_MS, SOLAP = 250, 0.5
ventanas, inicios = crear_ventanas(emg_prueba_f, FS, VENTANA_MS, SOLAP)
n_win = ventanas.shape[1]

# Etiqueta, ciclo y saturación por ventana
frac_activa = np.array([mascara_activa[i:i+n_win].mean() for i in inicios])
etiqueta = np.where(frac_activa >= 0.8, "actividad", np.where(frac_activa <= 0.05, "menor actividad", "transicion"))

inicios_contr = np.array([s for s, _ in contracciones])
ciclo = np.searchsorted(inicios_contr, inicios + n_win // 2, side="right")
ciclo = np.where(etiqueta == "menor actividad", np.searchsorted(inicios_contr, inicios, side="left"), ciclo - 1)
ciclo = np.clip(ciclo, 0, len(contracciones) - 1)

ventana_saturada = np.array([sat_prueba[i:i+n_win].any() for i in inicios])

print(f"Ventanas: {len(ventanas)} de {n_win} muestras ({VENTANA_MS} ms, solapamiento {int(SOLAP*100)} %)")
print(pd.Series(etiqueta).value_counts().to_string())
print(f"Ventanas con al menos una muestra saturada: {ventana_saturada.sum()}")

📘 **Explicación.** La señal filtrada se divide en ventanas de 250 ms con 50 % de solapamiento. Cada ventana se etiqueta a partir de la fracción que coincide con los segmentos de actividad detectados:

| Etiqueta | Criterio |
|---|---|
| menor actividad | ≤ 5 % de sus muestras dentro de un segmento activo |
| actividad | ≥ 80 % dentro de un segmento activo |
| transición | el resto (se excluye del análisis binario) |

El número de ventanas por clase y las que contienen muestras saturadas se imprimen en la celda. Las etiquetas son automáticas y no sustituyen una anotación manual.

---
# 5. Extracción de características clásicas (25 min)

Una **característica** es una función $f: \mathbb{R}^N \rightarrow \mathbb{R}$ que transforma una ventana de $N$ muestras en un número con significado. Las revisiones sistemáticas sobre reconocimiento de patrones mioeléctricos las agrupan en familias. Trabajaremos con cuatro.

### 5.1 Familia de amplitud / energía
*¿Cuánta actividad hay?* Crecen con el reclutamiento y la frecuencia de disparo.

| Característica | Fórmula | Intuición |
|---|---|---|
| **MAV** (*Mean Absolute Value*) | $\frac{1}{N}\sum_{i=1}^{N}\lvert x_i\rvert$ | Amplitud media rectificada |
| **RMS** (*Root Mean Square*) | $\sqrt{\frac{1}{N}\sum x_i^2}$ | Relacionada con la potencia de la señal |
| **iEMG** (EMG integrada) | $\sum \lvert x_i\rvert$ | Área bajo la señal rectificada |
| **VAR** | $\frac{1}{N-1}\sum (x_i-\bar{x})^2$ | Potencia (la sEMG filtrada tiene media ≈ 0) |

> ⚠️ RMS ≠ fuerza. La relación amplitud-fuerza depende del músculo, es no lineal y requiere normalización (p. ej., respecto de una contracción voluntaria máxima, %MVC) para comparar sujetos o sesiones.

### 5.2 Familia de complejidad / forma de onda
*¿Qué tan rápido y cuánto cambia la señal?* Forman el conjunto clásico propuesto por **Hudgins et al. (1993)** para control de prótesis. Las tres últimas usan un **umbral de ruido** $\varepsilon$ para no contar fluctuaciones del piso de ruido; lo tomaremos del referencia intra-registro: $\varepsilon = 3\,\sigma_{referencia intra-registro}$.

| Característica | Definición | Intuición |
|---|---|---|
| **WL** (*Waveform Length*) | $\sum_{i=1}^{N-1}\lvert x_{i+1}-x_i\rvert$ | "Longitud del trazo": combina amplitud y frecuencia |
| **ZC** (*Zero Crossings*) | nº de veces que $x$ cambia de signo con $\lvert x_{i}-x_{i+1}\rvert \ge \varepsilon$ | Estimador temporal de frecuencia |
| **SSC** (*Slope Sign Changes*) | nº de cambios de signo de la pendiente que superan $\varepsilon$ | Densidad de picos |
| **WAMP** (*Willison Amplitude*) | nº de veces que $\lvert x_{i+1}-x_i\rvert > \varepsilon$ | Relacionada con el disparo de unidades motoras |

### 5.3 Familia estadística (forma de la distribución)
*¿Se parecen las amplitudes a una campana de Gauss?*

| Característica | Fórmula | Interpretación |
|---|---|---|
| **Asimetría** (*skewness*) | $\frac{E[(x-\mu)^3]}{\sigma^3}$ | 0 = distribución simétrica |
| **Curtosis de Fisher** | $\frac{E[(x-\mu)^4]}{\sigma^4}-3$ | 0 = gaussiana; > 0 = colas pesadas / picos aislados; < 0 = distribución "aplanada" |

En contracciones con muchas UM, la sEMG tiende a ser cuasi-gaussiana (curtosis cercana a 0). Con pocas UM, o con artefactos, aparecen picos aislados (curtosis alta). La **saturación** recorta las colas y empuja la curtosis hacia valores **negativos**.

### 5.4 Familia espectral
*¿De qué "tonos" está hecha la señal?* A partir de la PSD $P(f)$ por Welch:

| Característica | Fórmula | Interpretación |
|---|---|---|
| **MNF** (frecuencia media) | $\dfrac{\sum f_j P_j}{\sum P_j}$ | Centro de masa del espectro |
| **MDF** (frecuencia mediana) | $f_{med}$ tal que $\sum_{f\le f_{med}} P = \tfrac12 \sum P$ | Divide la potencia en dos mitades; más robusta al ruido |
| **Potencia por bandas** | fracción de potencia en 20–80, 80–150 y 150–450 Hz | Distribución espectral gruesa |

> 💡 **Fatiga muscular:** al fatigarse, disminuye la velocidad de conducción de las fibras y los MUAPs se "ensanchan" → el espectro se desplaza a frecuencias **más bajas** → **MDF y MNF disminuyen**, mientras el RMS suele **aumentar**. Es uno de los indicadores de fatiga más usados.

In [ ]:
EPS = 3 * np.std(emg_base_f)   # umbral de ruido para ZC, SSC y WAMP
print(f"Umbral de ruido ε = {EPS*1000:.1f} µV")

def caracteristicas_temporales(x, eps=EPS):
    dx = np.diff(x)
    zc   = np.sum((x[:-1] * x[1:] < 0) & (np.abs(dx) >= eps))
    ssc  = np.sum(((dx[:-1] * dx[1:]) < 0) & ((np.abs(dx[:-1]) >= eps) | (np.abs(dx[1:]) >= eps)))
    return {
        "MAV":  np.mean(np.abs(x)),
        "RMS":  np.sqrt(np.mean(x**2)),
        "iEMG": np.sum(np.abs(x)),
        "VAR":  np.var(x, ddof=1),
        "WL":   np.sum(np.abs(dx)),
        "ZC":   zc,
        "SSC":  ssc,
        "WAMP": np.sum(np.abs(dx) > eps),
        "Asimetria": skew(x),
        "Curtosis":  kurtosis(x, fisher=True, bias=False),
    }

def caracteristicas_frecuenciales(x, fs=FS):
    f, p = signal.welch(x, fs=fs, nperseg=min(128, len(x)))
    banda = (f >= 20) & (f <= 450)
    f, p = f[banda], p[banda]
    p_tot = p.sum()
    acum = np.cumsum(p)
    pot = lambda lo, hi: p[(f >= lo) & (f < hi)].sum() / p_tot
    return {
        "MNF": np.sum(f * p) / p_tot,
        "MDF": f[np.searchsorted(acum, acum[-1] / 2)],
        "P_20_80":   pot(20, 80),
        "P_80_150":  pot(80, 150),
        "P_150_450": pot(150, 451),
    }

def extraer_caracteristicas(x, fs=FS):
    c = caracteristicas_temporales(x)
    c.update(caracteristicas_frecuenciales(x, fs))
    return c

# Prueba con una sola ventana de cada clase
i_rep = np.flatnonzero(etiqueta == "menor actividad")[3]
i_con = np.flatnonzero(etiqueta == "actividad")[3]
display(pd.DataFrame({"reposo": extraer_caracteristicas(ventanas[i_rep]),
                      "contraccion": extraer_caracteristicas(ventanas[i_con])}).round(4))

📘 **Explicación.** Primero se fijó el **umbral de ruido ε = 17.6 µV** (3 veces la desviación estándar del referencia intra-registro), que usan ZC, SSC y WAMP para no contar fluctuaciones pequeñas.

Luego se calcularon las 15 características para **una ventana de ejemplo** de cada clase. Comparando las columnas:
- **Amplitud:** el RMS pasa de 0.0076 a 0.0825 mV (unas 11 veces más). MAV, iEMG, VAR y WL suben de forma parecida.
- **Complejidad:** ZC pasa de 3 a 41, SSC de 3 a 51 y WAMP de 4 a 145.
- **Espectrales:** en este ejemplo la MDF *baja* (109 → 78 Hz). La potencia en 150–450 Hz pasa de 39 % a 9 %.

⚠️ Son **solo dos ventanas**. Para sacar conclusiones hay que mirar los promedios de todas las ventanas, que están en la celda siguiente.

In [ ]:
df = pd.DataFrame([extraer_caracteristicas(w) for w in ventanas])
CARACTERISTICAS = list(df.columns)
df.insert(0, "t_inicio_s", inicios / FS)
df["etiqueta"] = etiqueta
df["ciclo"] = ciclo
df["saturada"] = ventana_saturada

print("Matriz de características:", df[CARACTERISTICAS].shape, "(ventanas × características)")
display(df.head())
display(df.groupby("etiqueta")[CARACTERISTICAS].mean().T.round(4))

📘 **Explicación.** Se construye la matriz de características con una fila por ventana y las 15 características calculadas. La tabla de salida muestra los promedios por etiqueta estimada. Compara principalmente RMS, MAV y características espectrales. Recuerda que “menor actividad” y “actividad” se definieron a partir de la envolvente del mismo registro; por tanto, la separación observada es descriptiva y no una validación independiente.

### 5.5 Visualización temporal de las características
Observa cómo cada familia "responde" a las contracciones.

In [ ]:
mostrar = ["RMS", "WL", "ZC", "Curtosis", "MDF"]
fig, ax = plt.subplots(len(mostrar) + 1, 1, figsize=(14, 11), sharex=True)
ax[0].plot(t_prueba, emg_prueba_f, lw=0.3, color="gray")
ax[0].set_ylabel("EMG (mV)")
for a, c in zip(ax[1:], mostrar):
    a.plot(df["t_inicio_s"], df[c], lw=1, color=COLOR_UPCH)
    a.set_ylabel(c)
for a in ax:
    for s, e in contracciones:
        a.axvspan(s/FS, e/FS, color=COLOR_CONTRAC, alpha=0.08)
ax[-1].set_xlabel("Tiempo (s)")
plt.suptitle("Evolución de características por ventana (sombreado = actividad detectada)", y=1.0)
plt.tight_layout(); plt.show()

📘 **Explicación.** Cada fila muestra una característica a lo largo del tiempo; las franjas rosadas son las contracciones detectadas.

- **RMS y WL:** tienen picos nítidos que coinciden con cada contracción y valores casi nulos en el reposo. Son las curvas "más limpias".
- **ZC:** también sube en cada contracción, pero el contraste es menor y la curva es más irregular.
- **Curtosis:** muy ruidosa, con picos aislados que no siempre coinciden con las contracciones. A simple vista no separa bien las clases.
- **MDF:** oscila mucho de una ventana a otra (entre ~50 y ~130 Hz) y su relación con las contracciones no es evidente a simple vista.

### 5.6 ¿Qué característica separa mejor las clases?
Usaremos dos herramientas:
1. **Boxplots por clase**, para una inspección visual.
2. **Razón de Fisher** para cada característica: $J = \dfrac{(\mu_1-\mu_2)^2}{\sigma_1^2+\sigma_2^2}$. Mide la separación entre medias en relación con la dispersión. Cuanto mayor, más discriminante.

In [ ]:
df_ml = df[df["etiqueta"] != "transicion"].copy()
r, c = df_ml[df_ml.etiqueta == "menor actividad"], df_ml[df_ml.etiqueta == "actividad"]
fisher = ((r[CARACTERISTICAS].mean() - c[CARACTERISTICAS].mean())**2 /
          (r[CARACTERISTICAS].var() + c[CARACTERISTICAS].var())).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 2]})
fisher.plot.barh(ax=ax[0], color=COLOR_UPCH); ax[0].invert_yaxis()
ax[0].set_xscale("log"); ax[0].set_title("Razón de Fisher (escala log)")

seleccion = ["RMS", "ZC", "Curtosis", "MNF", "P_150_450"]
datos_z = (df_ml[seleccion] - df_ml[seleccion].mean()) / df_ml[seleccion].std()
pos = np.arange(len(seleccion))
for k, (lab, col) in enumerate([("menor actividad", COLOR_REPOSO), ("actividad", COLOR_CONTRAC)]):
    bp = ax[1].boxplot([datos_z[df_ml.etiqueta == lab][s] for s in seleccion],
                       positions=pos + (k - 0.5) * 0.35, widths=0.3, patch_artist=True, showfliers=False)
    for b in bp["boxes"]:
        b.set_facecolor(col); b.set_alpha(0.6)
ax[1].set_xticks(pos); ax[1].set_xticklabels(seleccion)
ax[1].set_title("Distribución por clase (valores estandarizados)  — azul: menor actividad, rojo: actividad")
plt.tight_layout(); plt.show()
print(fisher.round(2).to_string())

📘 **Explicación.** La razón de Fisher cuantifica la separación entre las dos etiquetas estimadas en relación con la dispersión de cada clase. El ranking se calcula a partir de los datos de este único registro. Una característica con Fisher alto separa mejor estas ventanas, pero esto no demuestra que clasifique correctamente registros nuevos ni sujetos diferentes. La comparación también depende del umbral automático utilizado para crear las etiquetas.

### 5.7 El efecto de la saturación sobre la curtosis
Comparamos ventanas de contracción **con y sin** muestras saturadas.

In [ ]:
con = df[df.etiqueta == "contraccion"]
resumen_sat = con.groupby("saturada")[["RMS", "Curtosis", "Asimetria", "MDF"]].agg(["mean", "count"]).round(3)
display(resumen_sat)
if con["saturada"].any():
    plt.figure(figsize=(6, 3.5))
    plt.boxplot([con[~con.saturada]["Curtosis"], con[con.saturada]["Curtosis"]], showfliers=False)
    plt.xticks([1, 2], ["sin saturación", "con saturación"])
    plt.axhline(0, color="k", lw=0.8, ls="--")
    plt.ylabel("Curtosis de Fisher"); plt.title("Curtosis en ventanas de contracción")
    plt.show()

📘 **Explicación.** La tabla separa las **ventanas de contracción** según tengan o no muestras saturadas:

| | Sin saturación (136 ventanas) | Con saturación (52 ventanas) |
|---|---|---|
| RMS medio | 0.173 mV | **0.604 mV** |
| Curtosis media | **3.72** | **1.20** |
| Asimetría media | 0.283 | 0.121 |
| MDF media | 69.5 Hz | 82.0 Hz |

- Las ventanas saturadas son **las de contracción más intensa**: su RMS medio es unas 3.5 veces mayor.
- Su **curtosis es mucho menor** (el boxplot lo muestra con claridad). Es lo que predice la teoría: el recorte elimina los valores extremos (las "colas" de la distribución) y la curtosis baja.
- Su MDF es mayor. Esto es coherente con que el recorte crea bordes abruptos, que añaden contenido de alta frecuencia. Pero en estas ventanas la saturación y la mayor intensidad ocurren a la vez, así que con estos datos no se pueden separar ambos efectos.

🔎 **Verificación adicional — ¿son redundantes las características de amplitud?**

La celda siguiente calcula la correlación entre MAV, RMS, iEMG, VAR y WL. Una correlación de 1 significa que dos características suben y bajan exactamente juntas.

In [ ]:

# Verificación adicional: ¿las características de amplitud son redundantes?
amp = ["MAV", "RMS", "iEMG", "VAR", "WL"]
print("Correlación de Pearson entre características de amplitud (ventanas de menor actividad y actividad):")
display(df_ml[amp].corr().round(3))


> ❓ **Preguntas de verificación**
> 1. ¿Qué características quedaron arriba en el ranking de Fisher? ¿ZC, SSC y WAMP miden solo frecuencia o también amplitud?
> 2. ¿Por qué RMS, MAV, iEMG y VAR pueden obtener rankings parecidos?
> 3. ¿Cómo cambia la curtosis en ventanas con saturación y qué harías en un estudio real?
>
> ✅ **Respuestas**
>
> 1. El ranking concreto aparece en la salida de la celda anterior. ZC, SSC y WAMP dependen de un umbral de amplitud; por ello combinan información de frecuencia de cambios y amplitud.
> 2. Porque son medidas relacionadas de la magnitud de la señal y pueden ser redundantes. La matriz de correlación de la celda siguiente permite comprobarlo con este registro.
> 3. La saturación recorta los picos y puede distorsionar curtosis, varianza y amplitud. En un estudio real se deben identificar y excluir o repetir las adquisiciones saturadas, además de ajustar la ganancia antes de registrar.

---
# 6. Del vector de características al clasificador (15 min)

### 6.1 El flujo clásico de reconocimiento de patrones
```
señal → preprocesamiento → ventanas → [ f1, f2, ..., fK ] → normalización → clasificador → decisión
```

Usaremos:
- **LDA** (*Linear Discriminant Analysis*): muy usado en control mioeléctrico; rápido, robusto y con pocos parámetros.
- **SVM con kernel RBF**: frontera de decisión no lineal.

La **normalización** (*z-score*) se ajusta **solo con los datos de entrenamiento** (por eso usamos un `Pipeline`).

### 6.2 ¡Cuidado con la fuga de datos (*data leakage*)!
Con solapamiento del 50 %, dos ventanas consecutivas **comparten la mitad de sus muestras**. Si una queda en entrenamiento y su vecina en prueba, el modelo "ya vio" parte de la prueba y el desempeño se **sobreestima**.

**Solución:** dividir por **grupos** (ciclos de reposo + contracción) con `GroupKFold`, de modo que ningún ciclo aparezca a la vez en entrenamiento y en prueba.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import KFold, GroupKFold, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, balanced_accuracy_score

X = df_ml[CARACTERISTICAS].values
y = (df_ml["etiqueta"] == "actividad").astype(int).values
grupos = df_ml["ciclo"].values

modelos = {
    "LDA": make_pipeline(StandardScaler(), LinearDiscriminantAnalysis()),
    "SVM-RBF": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale")),
}
conjuntos = {
    "Todas las características": CARACTERISTICAS,
    "Solo amplitud (RMS)": ["RMS"],
    "Sin amplitud (Curtosis, Asimetría, MNF, MDF, bandas)":
        ["Curtosis", "Asimetria", "MNF", "MDF", "P_20_80", "P_80_150", "P_150_450"],
}

cv_grupos = GroupKFold(n_splits=5)
filas = []
for nombre_conj, cols in conjuntos.items():
    Xc = df_ml[cols].values
    for nombre_mod, mod in modelos.items():
        s = cross_val_score(mod, Xc, y, cv=cv_grupos, groups=grupos, scoring="balanced_accuracy")
        filas.append({"Características": nombre_conj, "Modelo": nombre_mod,
                      "Exactitud balanceada": f"{s.mean():.3f} ± {s.std():.3f}"})
display(pd.DataFrame(filas))

📘 **Explicación.** Se entrenaron dos clasificadores (LDA y SVM) con tres conjuntos de características. Se evaluaron con **GroupKFold**: cada ciclo de reposo + contracción va completo al entrenamiento o a la prueba, nunca repartido. La métrica es la **exactitud balanceada**, que promedia el acierto de cada clase (0.5 equivale a adivinar al azar; 1.0 a acertar todo).

| Características | LDA | SVM |
|---|---|---|
| Todas (15) | 0.966 ± 0.023 | 0.972 ± 0.016 |
| Solo RMS | **0.768** ± 0.048 | 0.960 ± 0.020 |
| Sin amplitud (curtosis, asimetría, MNF, MDF, bandas) | 0.724 ± 0.047 | 0.737 ± 0.052 |

Dos cosas llaman la atención: el LDA con solo RMS rinde mucho peor que la SVM con la misma característica, y sin las características de amplitud el rendimiento cae, aunque sigue muy por encima del azar.

In [ ]:
# Matriz de confusión (LDA, todas las características, validación por grupos)
y_pred = cross_val_predict(modelos["LDA"], X, y, cv=cv_grupos, groups=grupos)
ConfusionMatrixDisplay(confusion_matrix(y, y_pred), display_labels=["menor actividad", "actividad"]).plot(cmap="Reds")
plt.title("LDA — GroupKFold (5 pliegues)"); plt.grid(False); plt.show()

# Demostración de la fuga: KFold aleatorio vs GroupKFold con las características SIN amplitud
cols_dificil = conjuntos["Sin amplitud (Curtosis, Asimetría, MNF, MDF, bandas)"]
Xd = df_ml[cols_dificil].values
mod = modelos["SVM-RBF"]
s_kf = cross_val_score(mod, Xd, y, cv=KFold(5, shuffle=True, random_state=0), scoring="balanced_accuracy")
s_gk = cross_val_score(mod, Xd, y, cv=cv_grupos, groups=grupos, scoring="balanced_accuracy")
print(f"SVM sin amplitud — KFold aleatorio (con fuga): {s_kf.mean():.3f}")
print(f"SVM sin amplitud — GroupKFold (sin fuga)    : {s_gk.mean():.3f}")

📘 **Explicación.** La matriz de confusión y las métricas de validación se calculan con las etiquetas automáticas del registro único. Aunque se dividan las ventanas por segmentos detectados, siguen procediendo de una sola sesión y un solo sujeto. Por ello, los resultados solo describen el comportamiento interno de este registro; no estiman generalización a nuevas personas ni a nuevas sesiones.

🔎 **Verificación adicional — para responder las preguntas del Bloque 6.**

La celda siguiente:
- (a) evalúa un clasificador que solo puede aprender **un umbral** sobre el RMS (árbol de decisión de profundidad 1);
- (b) prueba el LDA con **log(RMS)**;
- (c) describe la distribución del RMS en cada clase.

In [ ]:

# Verificación adicional para las preguntas del bloque 6
from sklearn.tree import DecisionTreeClassifier

# (a) ¿Basta un solo umbral en RMS? Un árbol de profundidad 1 aprende exactamente UN umbral
arbol = DecisionTreeClassifier(max_depth=1, random_state=0)
s_umbral = cross_val_score(arbol, df_ml[["RMS"]].values, y, cv=cv_grupos, groups=grupos, scoring="balanced_accuracy")
print(f"Un solo umbral en RMS (GroupKFold): {s_umbral.mean():.3f} ± {s_umbral.std():.3f}")

# (b) LDA con log(RMS) en lugar de RMS
s_log = cross_val_score(modelos["LDA"], np.log(df_ml[["RMS"]].values), y, cv=cv_grupos, groups=grupos,
                        scoring="balanced_accuracy")
s_rms = cross_val_score(modelos["LDA"], df_ml[["RMS"]].values, y, cv=cv_grupos, groups=grupos, scoring="balanced_accuracy")
print(f"LDA con RMS     : {s_rms.mean():.3f} ± {s_rms.std():.3f}")
print(f"LDA con log(RMS): {s_log.mean():.3f} ± {s_log.std():.3f}")

# (c) Asimetría del RMS en cada clase
for lab in ["menor actividad", "actividad"]:
    v = df_ml.loc[df_ml.etiqueta == lab, "RMS"]
    print(f"RMS en {lab:11s}: media {v.mean():.3f} mV | desv. est. {v.std():.3f} mV | asimetría {skew(v):.2f}")


> ❓ **Preguntas de verificación**
> 1. Para distinguir menor actividad de actividad en este registro, ¿bastaría un umbral en RMS? ¿Para qué serviría un vector de muchas características en tareas más complejas?
> 2. Sin las características de amplitud, ¿queda información para separar las etiquetas estimadas?
> 3. ¿Qué limitaciones tiene evaluar modelos con ventanas de un solo registro?
> 4. ¿Por qué una validación por grupos/sujetos es importante?
>
> ✅ **Respuestas**
>
> 1. Un umbral en RMS puede ser suficiente para una separación sencilla si las distribuciones están bien separadas. Muchas características son útiles para distinguir gestos diferentes, no solo actividad frente a menor actividad.
> 2. Puede quedar información espectral y de forma, pero debe comprobarse con las métricas calculadas; las etiquetas no son independientes de la señal.
> 3. Las ventanas de una misma grabación comparten sujeto, colocación de electrodos y condiciones. La validación puede ser optimista y no demuestra generalización.
> 4. Separar grupos evita que ventanas muy parecidas de una misma repetición aparezcan simultáneamente en entrenamiento y prueba. Para evaluar generalización entre personas se necesitan varios sujetos y validación por sujeto.

---
# 7. Deep learning: del espectrograma a una CNN (15 min)

### 7.1 Características *diseñadas* vs. características *aprendidas*
Hasta ahora **nosotros** decidimos qué calcular (RMS, ZC, MDF...): son características *hand-crafted*, basadas en conocimiento fisiológico. En deep learning, la red **aprende** sus propios filtros a partir de los datos.

| | Clásico (características + LDA/SVM) | Deep learning (CNN) |
|---|---|---|
| Entrada | Vector de ~15 números | Señal cruda (1D-CNN) o imagen tiempo-frecuencia (2D-CNN) |
| Conocimiento experto | Alto (se elige qué medir) | Bajo (la red lo descubre) |
| Datos necesarios | Pocos | Muchos (miles de ventanas, varios sujetos) |
| Interpretabilidad | Alta | Baja |
| Costo computacional | Mínimo | Alto (entrenamiento) |

### 7.2 La Transformada de Fourier de Tiempo Corto (STFT)
La FFT nos dice **qué** frecuencias hay, pero no **cuándo**. La STFT aplica la FFT sobre ventanas deslizantes $w[n]$:

$$
X(m, k) = \sum_{n} x[n]\,w[n - mH]\,e^{-j2\pi kn/N}
$$

El **espectrograma** $|X(m,k)|^2$ es una matriz *tiempo × frecuencia*: **una imagen**. Y las **CNN** son excelentes extrayendo patrones de imágenes.

> 💡 **Principio de incertidumbre tiempo-frecuencia:** una ventana larga da buena resolución en frecuencia y mala en tiempo, y viceversa. Con `nperseg = 64` a 1000 Hz → resolución de 15.6 Hz y 64 ms.

Otras representaciones usadas en la literatura: transformada wavelet continua (escalogramas), transformada de Hilbert-Huang y matrices multicanal en "imagen" (HD-sEMG).

In [ ]:
# Ventanas más largas (500 ms) para que el espectrograma tenga suficientes columnas
VENTANA_CNN_MS = 500
win_cnn, ini_cnn = crear_ventanas(emg_prueba_f, FS, VENTANA_CNN_MS, 0.5)
n_cnn = win_cnn.shape[1]
frac = np.array([mascara_activa[i:i+n_cnn].mean() for i in ini_cnn])
lab_cnn = np.where(frac >= 0.8, 1, np.where(frac <= 0.05, 0, -1))
ciclo_cnn = np.clip(np.where(lab_cnn == 0,
                             np.searchsorted(inicios_contr, ini_cnn, side="left"),
                             np.searchsorted(inicios_contr, ini_cnn + n_cnn//2, side="right") - 1),
                    0, len(contracciones) - 1)
ok = lab_cnn >= 0
win_cnn, lab_cnn, ciclo_cnn = win_cnn[ok], lab_cnn[ok], ciclo_cnn[ok]

def espectrograma(x, fs=FS, nperseg=64, noverlap=48):
    f, t, S = signal.stft(x, fs=fs, nperseg=nperseg, noverlap=noverlap, boundary=None)
    banda = (f >= 20) & (f <= 450)
    return f[banda], t, np.log10(np.abs(S[banda])**2 + 1e-10)

f_s, t_s, _ = espectrograma(win_cnn[0])
imagenes = np.stack([espectrograma(w)[2] for w in win_cnn]).astype(np.float32)
print(f"Imágenes: {imagenes.shape} (muestras, frecuencias, tiempos) | "
      f"menor actividad: {(lab_cnn==0).sum()}  actividad: {(lab_cnn==1).sum()}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for a, clase, nombre in zip(ax, [0, 1], ["menor actividad", "actividad"]):
    i = np.flatnonzero(lab_cnn == clase)[2]
    im = a.pcolormesh(t_s*1000, f_s, imagenes[i], shading="auto", cmap="magma")
    a.set_title(f"Espectrograma — {nombre}"); a.set_xlabel("ms"); a.set_ylabel("Hz")
    plt.colorbar(im, ax=a, label="log10 potencia")
plt.tight_layout(); plt.show()

# Espectrograma del registro completo: la "película" tiempo-frecuencia
f_full, t_full, S_full = signal.spectrogram(emg_prueba_f, fs=FS, nperseg=256, noverlap=192)
plt.figure(figsize=(14, 3.5))
plt.pcolormesh(t_full, f_full, 10*np.log10(S_full + 1e-12), shading="auto", cmap="magma")
plt.ylim(0, 500); plt.xlabel("Tiempo (s)"); plt.ylabel("Hz"); plt.colorbar(label="dB")
plt.title("Espectrograma del registro completo"); plt.show()

📘 **Explicación.** Para la CNN se usan ventanas de 500 ms y se convierten en espectrogramas (imágenes tiempo–frecuencia). El número de imágenes y la cantidad por clase se imprimen al ejecutar la celda. La comparación visual muestra diferencias de potencia entre ventanas etiquetadas como menor actividad y actividad. Como las etiquetas se derivan del mismo registro, esta separación es exploratoria.

### 7.3 Arquitectura de la CNN
Una red pequeña (≈ 1 300 parámetros) es suficiente para esta tarea y entrena en segundos en CPU:

```
Entrada (1 × F × T)
 → Conv2D 3×3, 8 filtros  → BatchNorm → ReLU → MaxPool 2×2      ← aprende patrones locales tiempo-frecuencia
 → Conv2D 3×3, 16 filtros → BatchNorm → ReLU → MaxPool 2×2      ← combina patrones en estructuras mayores
 → Global Average Pooling                                         ← resume cada mapa en un número
 → Lineal (16 → 2)                                                ← clasificador
```

Las capas convolucionales hacen el papel de **extractor de características**; la última capa lineal es el **clasificador**. Observa la analogía con el Bloque 6.

Usamos la **misma división por grupos** que antes para que la comparación sea justa. Cada ciclo entra completo a entrenamiento o a prueba, y la normalización se calcula solo con entrenamiento.

In [ ]:
try:
    import torch
    import torch.nn as nn
    TORCH_OK = True
except ImportError:
    TORCH_OK = False
    print("PyTorch no está disponible. En Colab viene preinstalado; localmente: pip install torch")

if TORCH_OK:
    torch.manual_seed(0)

    class CNNEspectrograma(nn.Module):
        def __init__(self, n_clases=2):
            super().__init__()
            self.extractor = nn.Sequential(
                nn.Conv2d(1, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(8, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
                nn.AdaptiveAvgPool2d(1),
            )
            self.clasificador = nn.Linear(16, n_clases)
        def forward(self, x):
            return self.clasificador(self.extractor(x).flatten(1))

    def entrenar_cnn(X_tr, y_tr, X_te, epocas=40, lr=1e-2):
        mu, sd = X_tr.mean(), X_tr.std()
        Xtr = torch.tensor((X_tr - mu) / sd).unsqueeze(1)
        Xte = torch.tensor((X_te - mu) / sd).unsqueeze(1)
        ytr = torch.tensor(y_tr, dtype=torch.long)
        modelo = CNNEspectrograma()
        opt = torch.optim.Adam(modelo.parameters(), lr=lr)
        perdida_fn = nn.CrossEntropyLoss()
        historial = []
        for _ in range(epocas):
            modelo.train()
            perm = torch.randperm(len(Xtr))
            for i in range(0, len(Xtr), 32):
                idx = perm[i:i+32]
                if len(idx) < 2:      # BatchNorm necesita al menos 2 muestras por lote
                    continue
                opt.zero_grad()
                loss = perdida_fn(modelo(Xtr[idx]), ytr[idx])
                loss.backward(); opt.step()
            historial.append(loss.item())
        modelo.eval()
        with torch.no_grad():
            pred = modelo(Xte).argmax(1).numpy()
        return pred, historial, modelo

    print("Parámetros entrenables:",
          sum(p.numel() for p in CNNEspectrograma().parameters() if p.requires_grad))

📘 **Explicación.** Se definió la red neuronal. Tiene **1 330 parámetros entrenables**, los números que la red ajusta durante el entrenamiento. Es una red muy pequeña (las CNN para imágenes suelen tener millones), elegida a propósito porque hay pocos datos.

In [ ]:
if TORCH_OK:
    resultados_cnn, y_true_all, y_pred_all = [], [], []
    for k, (tr, te) in enumerate(GroupKFold(n_splits=5).split(imagenes, lab_cnn, ciclo_cnn)):
        pred, hist, modelo_cnn = entrenar_cnn(imagenes[tr], lab_cnn[tr], imagenes[te])
        resultados_cnn.append(balanced_accuracy_score(lab_cnn[te], pred))
        y_true_all += list(lab_cnn[te]); y_pred_all += list(pred)
        print(f"Pliegue {k+1}: exactitud balanceada = {resultados_cnn[-1]:.3f}")
    print(f"\nCNN (espectrograma) — GroupKFold: {np.mean(resultados_cnn):.3f} ± {np.std(resultados_cnn):.3f}")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(hist, color=COLOR_UPCH); ax[0].set_title("Pérdida de entrenamiento (último pliegue)")
    ax[0].set_xlabel("época"); ax[0].set_ylabel("entropía cruzada")
    ConfusionMatrixDisplay(confusion_matrix(y_true_all, y_pred_all),
                           display_labels=["menor actividad", "actividad"]).plot(ax=ax[1], cmap="Reds", colorbar=False)
    ax[1].grid(False); ax[1].set_title("CNN — matriz de confusión agregada")
    plt.tight_layout(); plt.show()

📘 **Explicación.** La CNN se entrena usando los grupos de ventanas generados a partir de las contracciones detectadas en este registro. Los resultados por pliegue, promedio y matriz de confusión se imprimen al ejecutar la celda. No deben compararse con las métricas estáticas del solucionario anterior, que pertenecían a otros archivos. Una única grabación no permite afirmar que la CNN generalice a otros sujetos.

In [ ]:
# ¿Qué "mira" la red? Mapas de activación de los 8 filtros de la primera capa
if TORCH_OK:
    i = np.flatnonzero(lab_cnn == 1)[2]
    x_in = torch.tensor((imagenes[i] - imagenes.mean()) / imagenes.std()).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        mapas = modelo_cnn.extractor[:3](x_in)[0].numpy()
    fig, ax = plt.subplots(1, 8, figsize=(16, 2.4))
    for k in range(8):
        ax[k].imshow(mapas[k], origin="lower", aspect="auto", cmap="viridis")
        ax[k].set_title(f"filtro {k+1}", fontsize=8); ax[k].axis("off")
    plt.suptitle("Características aprendidas: activaciones de la 1.ª capa convolucional (ventana de actividad)", y=1.05)
    plt.show()

📘 **Explicación.** Cada recuadro muestra cómo responde uno de los 8 filtros de la primera capa ante una ventana de contracción. Las zonas claras indican dónde el filtro "se activa". En cada recuadro, abajo están las frecuencias bajas y arriba las altas.

Los filtros **no responden todos igual**. Por ejemplo, el filtro 7 se activa sobre todo en la parte baja (frecuencias bajas), mientras que los filtros 1 y 5 lo hacen en la parte superior. Cada filtro aprendió por sí solo a fijarse en una zona o patrón distinto del espectrograma: son **características aprendidas**, en lugar de diseñadas por nosotros. A diferencia del RMS o la MDF, no tienen un nombre ni una fórmula fisiológica clara, y esa es la pérdida de **interpretabilidad** del deep learning.

> ❓ **Preguntas de verificación**
> 1. ¿La CNN superó al LDA con características clásicas en las métricas obtenidas? Considerando costo e interpretabilidad, ¿cuál elegirías?
> 2. ¿Qué pasaría si probáramos en otra persona?
> 3. ¿Qué ventaja y desventaja tendría una 1D-CNN sobre señal cruda?
>
> ✅ **Respuestas**
>
> 1. La comparación debe hacerse con las métricas impresas al ejecutar ambos modelos. Con este único registro y etiquetas automáticas, cualquier diferencia es exploratoria; también deben considerarse el costo computacional y la interpretabilidad.
> 2. No se puede medir con los datos disponibles. La amplitud y el espectro dependen de la anatomía, tejido y posición de electrodos. Harían falta varios sujetos y validación separada por sujeto.
> 3. Una 1D-CNN evita elegir parámetros de STFT y aprende directamente de la señal, pero debe descubrir patrones temporales/frecuenciales y suele necesitar más datos.

---
# 8. Ejercicios

## Ejercicio 1 — Selección de características y tamaño de ventana (obligatorio)
1. Repite la extracción de características con ventanas de **100 ms** y **500 ms** (además de 250 ms).
2. Para cada tamaño, calcula la **razón de Fisher** de RMS, ZC, Curtosis y MDF.
3. Entrena un **LDA con solo las 3 características más discriminantes** (según Fisher) usando `GroupKFold`.
4. Elimina las ventanas saturadas y repite. ¿Cambia el resultado? ¿Cambia la curtosis promedio de la contracción?

**Entrega:** tabla (tamaño de ventana × característica) y una conclusión de 5–8 líneas.

*Completa las líneas marcadas con `# TODO`.*

In [ ]:

# ===== EJERCICIO 1 — SOLUCIÓN =====
def tabla_caracteristicas(x, ventana_ms, solap=0.5):
    w, ini = crear_ventanas(x, FS, ventana_ms, solap)
    n = w.shape[1]
    fr = np.array([mascara_activa[i:i+n].mean() for i in ini])
    et = np.where(fr >= 0.8, "actividad", np.where(fr <= 0.05, "menor actividad", "transicion"))
    d = pd.DataFrame([extraer_caracteristicas(v) for v in w])
    d["etiqueta"] = et
    d["ciclo"] = np.clip(np.where(et == "menor actividad",
                                  np.searchsorted(inicios_contr, ini, side="left"),
                                  np.searchsorted(inicios_contr, ini + n//2, side="right") - 1),
                         0, len(contracciones) - 1)
    d["saturada"] = [sat_prueba[i:i+n].any() for i in ini]
    d["saturada_corr"] = [((adc_prueba[i:i+n] <= 0) | (adc_prueba[i:i+n] >= 2**N_BITS - 2)).any() for i in ini]
    return d[d.etiqueta != "transicion"]

def razon_fisher(d, cols):
    a, b = d[d.etiqueta == "menor actividad"], d[d.etiqueta == "actividad"]
    return (a[cols].mean() - b[cols].mean())**2 / (a[cols].var() + b[cols].var())

# Paso 1 y 2: tabla de Fisher por tamaño de ventana
tablas = {vm: tabla_caracteristicas(emg_prueba_f, vm) for vm in [100, 250, 500]}
print("Número de ventanas usadas (sin transiciones):", {f"{k} ms": len(v) for k, v in tablas.items()})
display(pd.DataFrame({f"{vm} ms": razon_fisher(d, ["RMS", "ZC", "Curtosis", "MDF"])
                      for vm, d in tablas.items()}).round(2))

# Paso 3: top-3 por Fisher (250 ms) y LDA con GroupKFold
d250 = tablas[250]
top3 = razon_fisher(d250, CARACTERISTICAS).nlargest(3).index.tolist()
print("Top-3 características (250 ms):", top3)

def evaluar_lda(d, cols):
    yy = (d.etiqueta == "actividad").astype(int).values
    s = cross_val_score(make_pipeline(StandardScaler(), LinearDiscriminantAnalysis()),
                        d[cols].values, yy, cv=GroupKFold(5), groups=d.ciclo.values, scoring="balanced_accuracy")
    return s.mean(), s.std()

filas = []
for nombre, d in [("Todas las ventanas", d250),
                  ("Sin ventanas saturadas (criterio original)", d250[~d250.saturada]),
                  ("Sin ventanas saturadas (criterio corregido)", d250[~d250.saturada_corr])]:
    m, s = evaluar_lda(d, top3)
    curt = d.loc[d.etiqueta == "actividad", "Curtosis"].mean()
    filas.append({"Conjunto": nombre, "N ventanas": len(d),
                  "Contracciones": int((d.etiqueta == "actividad").sum()),
                  "LDA top-3 (exactitud balanceada)": f"{m:.3f} ± {s:.3f}",
                  "Curtosis media en actividad": round(curt, 2)})
display(pd.DataFrame(filas))


✅ **Respuesta — Ejercicio 1**

La celda anterior calcula la tabla de razón de Fisher para ventanas de 100, 250 y 500 ms, y evalúa el LDA con las tres características más discriminantes. Los valores se generan con el registro proporcionado y se muestran en pantalla.

**Conclusión que debe guiar la interpretación:**
- Las ventanas largas pueden estabilizar características espectrales, pero reducen la resolución temporal y el número de ejemplos.
- La selección de características depende de la señal y de las etiquetas estimadas.
- Las ventanas saturadas deben identificarse porque el recorte altera las características.
- La exactitud del LDA, si se calcula mediante grupos de segmentos de una sola grabación, no equivale a validación entre sujetos ni a una evaluación clínica independiente.
- La curtosis y demás estadísticas sensibles a picos deben interpretarse con especial cuidado cuando hay saturación.

## Ejercicio 2 — ¿Hay fatiga a lo largo de las contracciones? (tarea)
La teoría predice que con la fatiga **la MDF disminuye** y **el RMS aumenta**.

1. Para cada contracción detectada, calcula RMS, MNF y MDF sobre **toda la contracción**. Descarta 100 ms al inicio y al final para evitar las transiciones.
2. Grafica cada característica en función del número de contracción y ajusta una recta (`np.polyfit`).
3. Calcula la pendiente de la MDF y su correlación de Spearman con el número de contracción (`scipy.stats.spearmanr`).
4. Discute: ¿hay evidencia de fatiga? ¿Qué limitaciones tiene este protocolo para concluirlo? Piensa en la intensidad no controlada, en pocas repeticiones y en la saturación.

Opcional, para quien quiera profundizar en deep learning: entrena la CNN del Bloque 7 con ventanas de 250 ms y compara.

In [ ]:

# ===== EJERCICIO 2 — SOLUCIÓN =====
from scipy.stats import spearmanr

MARGEN = int(0.1 * FS)
filas = []
for k, (s, e) in enumerate(contracciones):
    seg = emg_prueba_f[s + MARGEN : e - MARGEN]
    if len(seg) < 0.3 * FS:
        continue
    c = extraer_caracteristicas(seg)
    filas.append({"actividad": k + 1, "inicio_s": round(s/FS, 2), "fin_s": round(e/FS, 2),
                  "RMS": c["RMS"], "MNF": c["MNF"], "MDF": c["MDF"]})
df_fatiga = pd.DataFrame(filas)
print(f"Contracciones analizadas: {len(df_fatiga)} | duración del registro: {len(emg_prueba_f)/FS:.2f} s")
display(df_fatiga.round(3))

unidades = {"RMS": ("µV", 1000), "MNF": ("Hz", 1), "MDF": ("Hz", 1)}
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
resumen = []
for a, nombre in zip(ax, ["RMS", "MNF", "MDF"]):
    xk, yk = df_fatiga["actividad"], df_fatiga[nombre]
    pend, inter = np.polyfit(xk, yk, 1)
    rho, p = spearmanr(xk, yk)
    u, esc = unidades[nombre]
    a.plot(xk, yk, "o", color=COLOR_UPCH); a.plot(xk, pend*xk + inter, "k--")
    a.set_title(f"{nombre}: pendiente = {pend*esc:.2f} {u}/actividad"); a.set_xlabel("n.º de actividad")
    resumen.append({"Característica": nombre, f"Pendiente": f"{pend*esc:.2f} {u}/contr.",
                    "rho de Spearman": round(rho, 2), "p-valor": round(p, 3)})
plt.tight_layout(); plt.show()
print("Con todas las contracciones:"); display(pd.DataFrame(resumen))

# Sensibilidad: repetir sin la última actividad (comprobar en la tabla si termina junto con el registro)
d2 = df_fatiga.iloc[:-1]
res2 = []
for nombre in ["RMS", "MNF", "MDF"]:
    pend = np.polyfit(d2["actividad"], d2[nombre], 1)[0]
    rho, p = spearmanr(d2["actividad"], d2[nombre])
    u, esc = unidades[nombre]
    res2.append({"Característica": nombre, "Pendiente": f"{pend*esc:.2f} {u}/contr.",
                 "rho de Spearman": round(rho, 2), "p-valor": round(p, 3)})
print("Sin la última actividad:"); display(pd.DataFrame(res2))


✅ **Respuesta — Ejercicio 2**

La tabla y las gráficas de RMS, MNF y MDF se calculan a partir de los segmentos detectados en el registro. **No se debe copiar aquí una tabla de resultados del solucionario anterior**, porque correspondía a una grabación distinta.

**¿Hay evidencia suficiente para afirmar fatiga? No puede concluirse con este registro por sí solo.**

- La fatiga puede asociarse con una disminución de MDF/MNF y un aumento de RMS, pero estos cambios también dependen de la intensidad de contracción y de la colocación de electrodos.
- La grabación es breve y no se documenta una fuerza de contracción controlada.
- Las pausas y la saturación pueden afectar la interpretación.
- Una correlación o pendiente calculada sobre pocas contracciones detectadas es exploratoria y no demuestra causalidad.

Para un protocolo de fatiga se requerirían contracciones sostenidas o repetidas a intensidad controlada, más repeticiones, descansos estandarizados y control de la saturación.

---
# 9. Cierre y síntesis

### Lo que hicimos
| Etapa | Concepto clave |
|---|---|
| Fisiología | La sEMG es la suma de MUAPs; amplitud y espectro reflejan el reclutamiento y la frecuencia de disparo |
| Adquisición | Amplificación diferencial, Fs ≥ 2·f_max, conversión ADC → mV, **la saturación no se repara** |
| Preprocesamiento | Pasa-banda 20–450 Hz + notch 60 Hz, filtrado de fase cero |
| Segmentación | Envolvente RMS, umbral con reposo **en las mismas condiciones**, ventanas de 100–500 ms |
| Características | Amplitud (RMS, MAV), complejidad (WL, ZC, SSC, WAMP), estadísticas (curtosis, asimetría), espectrales (MNF, MDF) |
| Clasificación | Normalización dentro del pipeline, **validación por grupos** para evitar fuga |
| Deep learning | STFT → espectrograma → CNN que **aprende** sus características |

### Para pensar
- ¿Qué características usarías para una prótesis que debe decidir en menos de 300 ms?
- ¿Cómo cambiaría el análisis con **varios canales** (varios músculos) en lugar de uno?
- ¿Qué haría falta para que el modelo funcione en **otro sujeto**?

### Entregables
1. Este cuaderno ejecutado, con las preguntas de verificación respondidas en celdas de texto.
2. Ejercicio 1 completo (tabla y conclusión).
3. Ejercicio 2 (gráficas de RMS, MNF y MDF con su interpretación).
4. Archivo `caracteristicas_emg.csv` (se genera en la celda siguiente).

### Referencias recomendadas
- De Luca, C. J. (1997). *The use of surface electromyography in biomechanics.* Journal of Applied Biomechanics, 13(2), 135–163.
- Hermens, H. J., Freriks, B., Disselhorst-Klug, C., & Rau, G. (2000). *Development of recommendations for SEMG sensors and sensor placement procedures (SENIAM).* Journal of Electromyography and Kinesiology, 10(5), 361–374.
- Hudgins, B., Parker, P., & Scott, R. N. (1993). *A new strategy for multifunction myoelectric control.* IEEE Transactions on Biomedical Engineering, 40(1), 82–94.
- Phinyomark, A., Phukpattaranont, P., & Limsakul, C. (2012). *Feature reduction and selection for EMG signal classification.* Expert Systems with Applications, 39(8), 7420–7431.
- Atzori, M., Cognolato, M., & Müller, H. (2016). *Deep learning with convolutional neural networks applied to electromyography data: a resource for the classification of movements for prosthetic hands.* Frontiers in Neurorobotics, 10, 9.
- PLUX. *BITalino (r)evolution — EMG Sensor Datasheet* (fórmula de conversión de unidades).

✅ **Respuestas a "Para pensar"** *(conceptuales: se apoyan en la teoría; el registro único no permite probar generalización)*

- **Prótesis con decisión en menos de 300 ms:** se necesitan características rápidas y estables con ventanas cortas, como RMS/MAV, WL, ZC, SSC y WAMP. La elección final debe equilibrar rapidez y capacidad discriminante.
- **Varios canales (varios músculos):** se calculan características para cada canal y se concatenan. Esto permite distinguir gestos por qué músculos se activan y cuánto.
- **Que el modelo funcione en otra persona:** se necesitan registros de varios sujetos, colocación estandarizada de electrodos (SENIAM), normalización de amplitud (por ejemplo, respecto de %MVC) y validación por sujeto. Este laboratorio no puede medir generalización porque solo contiene una sesión y un canal.

In [ ]:
df.to_csv("caracteristicas_emg_registro_unico.csv", index=False)
print("Archivo generado: caracteristicas_emg_registro_unico.csv", df.shape)
if EN_COLAB:
    from google.colab import files
    files.download("caracteristicas_emg_registro_unico.csv")

📘 **Explicación.** Se guarda la matriz de características en `caracteristicas_emg.csv`. El número de filas depende de las ventanas generadas a partir de la duración del registro; las columnas incluyen tiempo, características, etiqueta estimada, grupo y saturación. En Colab, el archivo se descarga automáticamente.